# Data Layer Raw Collection Notebook

目标：在 `NUS` conda 内核下运行，复用已有 HDB resale raw 数据，新增公共免费数据源：
- OneMap：HDB 地址地理编码（替代 Google Geocoding）
- MOE General Information of Schools：学校基础信息
- NEA Hawker Centres：食阁地理信息
- LTA / 公共交通节点数据（DataMall API key 可选，OneMap 关键词检索兜底）

输出目录：
- `01_data_layer/raw/google_geo/`（保留目录名，内容改为公共地理数据）
- `01_data_layer/raw/schools/`
- `01_data_layer/raw/logs/`

In [2]:
import os
import sys
import json
import time
import random
import hashlib
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# Section 1: 环境与内核配置（Conda: NUS）
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

try:
    import importlib.metadata as importlib_metadata
except Exception:  # pragma: no cover
    import importlib_metadata  # type: ignore

for pkg in ['pandas', 'requests', 'beautifulsoup4', 'googlemaps']:
    try:
        print(f'{pkg} version:', importlib_metadata.version(pkg))
    except Exception:
        print(f'{pkg} version: not installed')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)
print('Random seed:', SEED)

Python executable: /opt/homebrew/anaconda3/envs/nus/bin/python
Python version: 3.12.9
pandas version: 3.0.0
requests version: 2.32.5
beautifulsoup4 version: 4.14.3
googlemaps version: not installed
Random seed: 42


## 2) 项目路径与 raw 目录约定

In [3]:
NOTEBOOK_PATH = Path.cwd() / '01_data_layer' / 'pipelines' / 'raw_data_collection.ipynb'
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / '01_data_layer').exists():
    # Notebook may run with cwd at notebook folder
    PROJECT_ROOT = Path.cwd().parents[1]

RAW_ROOT = PROJECT_ROOT / '01_data_layer' / 'raw'
HDB_RAW_DIR = RAW_ROOT / 'ResaleFlatPrices'
GEO_RAW_DIR = RAW_ROOT / 'google_geo'
SCHOOL_RAW_DIR = RAW_ROOT / 'schools'
LOG_DIR = RAW_ROOT / 'logs'

for p in [RAW_ROOT, HDB_RAW_DIR, GEO_RAW_DIR, SCHOOL_RAW_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_ROOT =', RAW_ROOT)
print('HDB_RAW_DIR =', HDB_RAW_DIR)
print('GEO_RAW_DIR =', GEO_RAW_DIR)
print('SCHOOL_RAW_DIR =', SCHOOL_RAW_DIR)
print('LOG_DIR =', LOG_DIR)

PROJECT_ROOT = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens
RAW_ROOT = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw
HDB_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/ResaleFlatPrices
GEO_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/google_geo
SCHOOL_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/schools
LOG_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/logs


## 3) 加载现有 HDB resale 原始数据（只读校验）

In [4]:
required_cols = {
    'month', 'town', 'block', 'street_name', 'flat_type', 'storey_range',
    'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price'
}

hdb_files = sorted(HDB_RAW_DIR.glob('*.csv'))
assert hdb_files, f'No HDB CSV found in {HDB_RAW_DIR}'
print('Found HDB files:', len(hdb_files))
for f in hdb_files:
    print('-', f.name)

df_list = []
for fp in hdb_files:
    t = pd.read_csv(fp)
    missing = required_cols - set(t.columns)
    if missing:
        raise ValueError(f'{fp.name} missing columns: {missing}')
    t['source_file'] = fp.name
    df_list.append(t)

hdb_raw = pd.concat(df_list, ignore_index=True)
hdb_raw['month_dt'] = pd.to_datetime(hdb_raw['month'], errors='coerce')
hdb_2015 = hdb_raw[hdb_raw['month_dt'] >= pd.Timestamp('2015-01-01')].copy()

hdb_2015['address_query'] = (
    hdb_2015['block'].astype(str).str.strip() + ' ' +
    hdb_2015['street_name'].astype(str).str.strip() + ', Singapore'
)

address_df = hdb_2015[['block', 'street_name', 'town', 'address_query']].drop_duplicates().reset_index(drop=True)
address_df['source_id'] = address_df.apply(
    lambda r: hashlib.md5(f"{r['block']}|{r['street_name']}|{r['town']}".encode('utf-8')).hexdigest(),
    axis=1
)

print('Total HDB rows (all years):', len(hdb_raw))
print('Total HDB rows (2015+):', len(hdb_2015))
print('Unique addresses for geo collection (2015+):', len(address_df))
address_df.head()

Found HDB files: 3
- Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv
- Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv
- Resale flat prices based on registration date from Jan-2017 onwards.csv
Total HDB rows (all years): 315627
Total HDB rows (2015+): 263424
Unique addresses for geo collection (2015+): 9710


,block,street_name,town,address_query,source_id
0,174,ANG MO KIO AVE 4,ANG MO KIO,"174 ANG MO KIO AVE 4, Singapore",f10a0ad9f92b5fed1af5d220a2ca1e57
1,541,ANG MO KIO AVE 10,ANG MO KIO,"541 ANG MO KIO AVE 10, Singapore",5ba09c4a5cb55a06441120282b2df0de
2,163,ANG MO KIO AVE 4,ANG MO KIO,"163 ANG MO KIO AVE 4, Singapore",00efd671b0f8ae3a83c81c58597802ea
3,446,ANG MO KIO AVE 10,ANG MO KIO,"446 ANG MO KIO AVE 10, Singapore",6165c56500e0d3f53d30eb9a00099925
4,557,ANG MO KIO AVE 10,ANG MO KIO,"557 ANG MO KIO AVE 10, Singapore",64cec601457d63791c41921d07ef3e14


## 4) 公共数据源凭据与请求客户端初始化（Data.gov/OneMap/LTA）

In [5]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_env_file(PROJECT_ROOT / '.env')
load_env_file(PROJECT_ROOT / '01_data_layer' / '.env')

# Optional keys (pipeline can run partially without them)
DATA_GOV_SG_API_KEY = os.getenv('DATA_GOV_SG_API_KEY')
ONEMAP_API_KEY = os.getenv('ONEMAP_API_KEY')
DATAMALL_API_KEY = os.getenv('DATAMALL_API_KEY')

session = requests.Session()
retry = Retry(
    total=6,
    connect=6,
    read=6,
    backoff_factor=1.2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(['GET'])
)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

REQUEST_TIMEOUT = 25
QPS_LIMIT = 5
MIN_INTERVAL = 1.0 / QPS_LIMIT
_last_call_ts = 0.0

def rate_limit_sleep():
    global _last_call_ts
    now = time.time()
    elapsed = now - _last_call_ts
    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed)
    _last_call_ts = time.time()

print('DATA_GOV_SG_API_KEY found:', bool(DATA_GOV_SG_API_KEY))
print('ONEMAP_API_KEY found:', bool(ONEMAP_API_KEY))
print('DATAMALL_API_KEY found:', bool(DATAMALL_API_KEY))
print('Note: OneMap geocode/search can still return results without API token in some cases.')

DATA_GOV_SG_API_KEY found: False
ONEMAP_API_KEY found: True
DATAMALL_API_KEY found: False
Note: OneMap geocode/search can still return results without API token in some cases.


## 5) 公共地理与学校数据采集（OneMap + MOE + NEA + LTA）

In [24]:

DATASTORE_ENDPOINT = 'https://data.gov.sg/api/action/datastore_search'
ONEMAP_SEARCH_ENDPOINT = 'https://www.onemap.gov.sg/api/common/elastic/search'
DATAMALL_BUS_STOPS_ENDPOINT = 'https://datamall2.mytransport.sg/ltaodataservice/BusStops'

MOE_SCHOOLS_RESOURCE_ID = 'd_688b934f82c1059ed0a6993d2a829089'
NEA_HAWKER_RESOURCE_ID = 'd_4a086da0a5553be1d89383cd90d07ecd'

ONEMAP_GEO_CHECKPOINT_FILE = GEO_RAW_DIR / 'onemap_geocode_checkpoint.csv'
ONEMAP_GEO_RAW_RESP_FILE = GEO_RAW_DIR / 'onemap_geocode_raw_responses.jsonl'

ONEMAP_MAX_ADDRESSES = int(os.getenv('ONEMAP_MAX_ADDRESSES', '0'))  # 0 means all
DATASTORE_PAGE_SIZE = int(os.getenv('DATASTORE_PAGE_SIZE', '5000'))


def safe_get_json(url: str, params: dict | None = None, headers: dict | None = None) -> dict:
    rate_limit_sleep()
    try:
        r = session.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
        try:
            payload = r.json() if r.content else {}
        except Exception:
            payload = {}
        return {'http_status': r.status_code, 'payload': payload}
    except requests.exceptions.ConnectionError as e:
        print(f'  [safe_get_json] ConnectionError for {url}: {e}')
        return {'http_status': 0, 'payload': {}, 'error': f'ConnectionError:{e}'}
    except requests.exceptions.Timeout as e:
        print(f'  [safe_get_json] Timeout for {url}: {e}')
        return {'http_status': 0, 'payload': {}, 'error': f'Timeout:{e}'}
    except Exception as e:
        print(f'  [safe_get_json] Unexpected error for {url}: {e}')
        return {'http_status': 0, 'payload': {}, 'error': f'{type(e).__name__}:{e}'}


def fetch_datagov_records(resource_id: str, page_size: int = DATASTORE_PAGE_SIZE) -> tuple[pd.DataFrame, pd.DataFrame]:
    headers = {'x-api-key': DATA_GOV_SG_API_KEY} if DATA_GOV_SG_API_KEY else None
    offset = 0
    all_rows = []
    audit_rows = []

    while True:
        params = {'resource_id': resource_id, 'limit': page_size, 'offset': offset}
        resp = safe_get_json(DATASTORE_ENDPOINT, params=params, headers=headers)
        payload = resp.get('payload', {})

        if resp.get('error'):
            audit_rows.append({'resource_id': resource_id, 'offset': offset, 'row_count': 0,
                                'http_status': resp['http_status'], 'status': f"network_error:{resp['error'][:120]}"})
            break

        if payload.get('success') is True and 'result' in payload:
            rows = payload['result'].get('records', [])
            all_rows.extend(rows)
            audit_rows.append({'resource_id': resource_id, 'offset': offset, 'row_count': len(rows), 'http_status': resp['http_status'], 'status': 'ok'})
            if len(rows) < page_size:
                break
            offset += page_size
            continue

        code = payload.get('code')
        if code == 24:
            audit_rows.append({'resource_id': resource_id, 'offset': offset, 'row_count': 0, 'http_status': resp['http_status'], 'status': 'rate_limited'})
            time.sleep(10)
            continue

        audit_rows.append({'resource_id': resource_id, 'offset': offset, 'row_count': 0, 'http_status': resp['http_status'], 'status': f"error:{payload.get('errorMsg', payload.get('name', 'unknown'))}"})
        break

    return pd.DataFrame(all_rows), pd.DataFrame(audit_rows)


def onemap_search(search_val: str, page_num: int = 1) -> dict:
    params = {
        'searchVal': search_val,
        'returnGeom': 'Y',
        'getAddrDetails': 'Y',
        'pageNum': page_num,
    }
    headers = {'Authorization': ONEMAP_API_KEY} if ONEMAP_API_KEY else None
    return safe_get_json(ONEMAP_SEARCH_ENDPOINT, params=params, headers=headers)


def _restore_done_ids_from_raw_jsonl(raw_file: Path) -> set[str]:
    done = set()
    if not raw_file.exists():
        return done
    with raw_file.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                sid = obj.get('source_id')
                if obj.get('response', {}).get('payload', {}).get('results', []) and sid:
                    done.add(str(sid))
            except Exception:
                continue
    return done


def _reconstruct_geo_from_jsonl(raw_file: Path) -> pd.DataFrame:
    """Rebuild geocoded DataFrame from raw JSONL responses."""
    seen = set()
    rows = []
    if not raw_file.exists():
        return pd.DataFrame()
    with raw_file.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                sid = obj.get('source_id', '')
                if sid in seen:
                    continue  # keep first successful entry per source_id
                collected_at = obj.get('collected_at', '')
                resp = obj.get('response', {})
                payload = resp.get('payload', {})
                results = payload.get('results', [])
                if results:
                    seen.add(sid)
                    best = results[0]
                    rows.append({
                        'source_id': sid,
                        'requested_address': obj.get('queried_address', ''),
                        'resolved_address': best.get('ADDRESS'),
                        'postal': best.get('POSTAL'),
                        'lat': best.get('LATITUDE'),
                        'lng': best.get('LONGITUDE'),
                        'x': best.get('X'),
                        'y': best.get('Y'),
                        'searchval': best.get('SEARCHVAL'),
                        'http_status': resp.get('http_status'),
                        'one_map_error': payload.get('error'),
                        'collected_at': collected_at,
                    })
            except Exception:
                continue
    return pd.DataFrame(rows)


def _write_checkpoint(existing_df: pd.DataFrame, processed_ids: list[str]) -> None:
    add_df = pd.DataFrame({'source_id': processed_ids}) if processed_ids else pd.DataFrame(columns=['source_id'])
    checkpoint = pd.concat([existing_df, add_df], ignore_index=True).drop_duplicates('source_id')
    checkpoint.to_csv(ONEMAP_GEO_CHECKPOINT_FILE, index=False)


def collect_onemap_hdb_geocode(address_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    existing = pd.read_csv(ONEMAP_GEO_CHECKPOINT_FILE) if ONEMAP_GEO_CHECKPOINT_FILE.exists() else pd.DataFrame(columns=['source_id'])
    done_ids = set(existing['source_id'].astype(str).tolist()) if len(existing) else set()

    # Fallback resume: recover successfully-geocoded ids from raw JSONL
    if not done_ids and ONEMAP_GEO_RAW_RESP_FILE.exists():
        recovered = _restore_done_ids_from_raw_jsonl(ONEMAP_GEO_RAW_RESP_FILE)
        if recovered:
            done_ids = recovered
            existing = pd.DataFrame({'source_id': sorted(done_ids)})
            _write_checkpoint(existing, [])
            print('Recovered done_ids from raw jsonl:', len(done_ids))

    to_run = address_frame[~address_frame['source_id'].astype(str).isin(done_ids)].copy()
    if ONEMAP_MAX_ADDRESSES > 0:
        to_run = to_run.head(ONEMAP_MAX_ADDRESSES)

    print('Addresses pending for OneMap geocode:', len(to_run))

    # If everything is already geocoded, reconstruct the DataFrame from the JSONL log
    if len(to_run) == 0 and ONEMAP_GEO_RAW_RESP_FILE.exists():
        print('All addresses already geocoded — reconstructing from JSONL...')
        geo_df = _reconstruct_geo_from_jsonl(ONEMAP_GEO_RAW_RESP_FILE)
        addr_map = address_frame.set_index('source_id')['address_query'].to_dict()
        geo_df['requested_address'] = geo_df['source_id'].map(addr_map).fillna(geo_df['requested_address'])
        audit_df = pd.DataFrame([{'source_id': r, 'requested_address': addr_map.get(r, ''), 'status': 'reconstructed'} for r in geo_df['source_id']])
        print(f'Reconstructed {len(geo_df)} geocoded rows from JSONL.')
        return geo_df, audit_df

    geo_rows = []
    audit_rows = []
    processed_ids = []

    for i, row in to_run.iterrows():
        sid = str(row['source_id'])
        addr = row['address_query']
        # OneMap returns 0 results when ", Singapore" is appended — strip it
        addr_query = re.sub(r',?\s*Singapore\s*$', '', addr, flags=re.IGNORECASE).strip()
        ts = datetime.utcnow().isoformat()

        try:
            resp = onemap_search(addr_query, page_num=1)
            payload = resp.get('payload', {})
            results = payload.get('results', [])

            with ONEMAP_GEO_RAW_RESP_FILE.open('a', encoding='utf-8') as f:
                f.write(json.dumps({'source_id': sid, 'queried_address': addr_query,
                                    'response': resp, 'collected_at': ts}, ensure_ascii=False) + '\n')

            if results:
                best = results[0]
                geo_rows.append({
                    'source_id': sid,
                    'requested_address': addr,
                    'resolved_address': best.get('ADDRESS'),
                    'postal': best.get('POSTAL'),
                    'lat': best.get('LATITUDE'),
                    'lng': best.get('LONGITUDE'),
                    'x': best.get('X'),
                    'y': best.get('Y'),
                    'searchval': best.get('SEARCHVAL'),
                    'http_status': resp.get('http_status'),
                    'one_map_error': payload.get('error'),
                    'collected_at': ts,
                })
                audit_rows.append({'source_id': sid, 'requested_address': addr, 'status': 'success'})
            else:
                audit_rows.append({'source_id': sid, 'requested_address': addr, 'status': 'no_result'})

        except Exception as e:
            audit_rows.append({'source_id': sid, 'requested_address': addr, 'status': f'error:{type(e).__name__}'})

        processed_ids.append(sid)

        if len(processed_ids) % 200 == 0:
            _write_checkpoint(existing, processed_ids)
            print(f'Checkpoint saved at {len(processed_ids)} newly processed addresses...')

    geo_df = pd.DataFrame(geo_rows)
    audit_df = pd.DataFrame(audit_rows)

    if len(processed_ids):
        _write_checkpoint(existing, processed_ids)

    print('OneMap geocode rows:', len(geo_df))
    print('OneMap geocode audit rows:', len(audit_df))
    return geo_df, audit_df


def collect_onemap_transit_nodes() -> pd.DataFrame:
    queries = [
        ('mrt_station', 'MRT STATION'),
        ('lrt_station', 'LRT STATION'),
        ('bus_interchange', 'BUS INTERCHANGE'),
    ]
    rows = []

    for node_type, q in queries:
        for page in range(1, 6):
            resp = onemap_search(q, page_num=page)
            payload = resp.get('payload', {})
            results = payload.get('results', [])
            if not results:
                break
            for r in results:
                rows.append({
                    'node_type': node_type,
                    'search_query': q,
                    'searchval': r.get('SEARCHVAL'),
                    'address': r.get('ADDRESS'),
                    'postal': r.get('POSTAL'),
                    'lat': r.get('LATITUDE'),
                    'lng': r.get('LONGITUDE'),
                    'x': r.get('X'),
                    'y': r.get('Y'),
                    'source': 'OneMap',
                })

    return pd.DataFrame(rows)


def collect_datamall_bus_stops() -> pd.DataFrame:
    if not DATAMALL_API_KEY:
        return pd.DataFrame()

    headers = {'AccountKey': DATAMALL_API_KEY, 'accept': 'application/json'}
    skip = 0
    rows = []

    while True:
        rate_limit_sleep()
        r = session.get(DATAMALL_BUS_STOPS_ENDPOINT, headers=headers, params={'$skip': skip}, timeout=REQUEST_TIMEOUT)
        payload = r.json() if r.content else {}
        batch = payload.get('value', [])
        rows.extend(batch)
        if len(batch) < 500:
            break
        skip += 500

    return pd.DataFrame(rows)


def normalize_geo(onemap_geo_df: pd.DataFrame, transit_df: pd.DataFrame, hawker_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    out_geo = onemap_geo_df.copy()
    if len(out_geo):
        out_geo['lat'] = pd.to_numeric(out_geo['lat'], errors='coerce').round(6)
        out_geo['lng'] = pd.to_numeric(out_geo['lng'], errors='coerce').round(6)
        out_geo = out_geo.sort_values('collected_at').drop_duplicates('source_id', keep='last')

    out_transit = transit_df.copy()
    if len(out_transit):
        out_transit['lat'] = pd.to_numeric(out_transit['lat'], errors='coerce').round(6)
        out_transit['lng'] = pd.to_numeric(out_transit['lng'], errors='coerce').round(6)
        out_transit = out_transit.drop_duplicates(['node_type', 'searchval', 'address'])

    out_hawker = hawker_df.copy()
    if len(out_hawker):
        for lat_col in ['latitude', 'lat', 'LATITUDE']:
            if lat_col in out_hawker.columns:
                out_hawker['lat'] = pd.to_numeric(out_hawker[lat_col], errors='coerce')
                break
        for lng_col in ['longitude', 'lng', 'LONGITUDE', 'longtitude']:
            if lng_col in out_hawker.columns:
                out_hawker['lng'] = pd.to_numeric(out_hawker[lng_col], errors='coerce')
                break
        out_hawker['lat'] = pd.to_numeric(out_hawker.get('lat'), errors='coerce').round(6)
        out_hawker['lng'] = pd.to_numeric(out_hawker.get('lng'), errors='coerce').round(6)

    return out_geo, out_transit, out_hawker

In [7]:
def normalize_geo(onemap_geo_df: pd.DataFrame, transit_df: pd.DataFrame, hawker_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    out_geo = onemap_geo_df.copy()
    if len(out_geo):
        out_geo['lat'] = pd.to_numeric(out_geo['lat'], errors='coerce').round(6)
        out_geo['lng'] = pd.to_numeric(out_geo['lng'], errors='coerce').round(6)
        out_geo = out_geo.sort_values('collected_at').drop_duplicates('source_id', keep='last')

    out_transit = transit_df.copy()
    if len(out_transit):
        out_transit['lat'] = pd.to_numeric(out_transit['lat'], errors='coerce').round(6)
        out_transit['lng'] = pd.to_numeric(out_transit['lng'], errors='coerce').round(6)
        out_transit = out_transit.drop_duplicates(['node_type', 'searchval', 'address'])

    out_hawker = hawker_df.copy()
    if len(out_hawker):
        for lat_col in ['latitude', 'lat', 'LATITUDE']:
            if lat_col in out_hawker.columns:
                out_hawker['lat'] = pd.to_numeric(out_hawker[lat_col], errors='coerce')
                break
        for lng_col in ['longitude', 'lng', 'LONGITUDE', 'longtitude']:
            if lng_col in out_hawker.columns:
                out_hawker['lng'] = pd.to_numeric(out_hawker[lng_col], errors='coerce')
                break
        out_hawker['lat'] = pd.to_numeric(out_hawker.get('lat'), errors='coerce').round(6)
        out_hawker['lng'] = pd.to_numeric(out_hawker.get('lng'), errors='coerce').round(6)

    return out_geo, out_transit, out_hawker

## 7) MOE / NEA / LTA 数据采集与学校数据处理

In [8]:
def collect_moe_school_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    schools_df, audit_df = fetch_datagov_records(MOE_SCHOOLS_RESOURCE_ID)
    if len(schools_df):
        schools_df['source_dataset'] = 'MOE General Information of Schools'
    return schools_df, audit_df


def collect_nea_hawker_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    hawker_df, audit_df = fetch_datagov_records(NEA_HAWKER_RESOURCE_ID)
    if len(hawker_df):
        hawker_df['source_dataset'] = 'NEA Hawker Centres'
    return hawker_df, audit_df


def process_school_dataset(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()
    # normalize common textual columns if present
    for c in out.columns:
        if out[c].dtype == 'object':
            out[c] = out[c].astype(str).str.strip()

    # derive school_level when possible
    candidate_cols = [c for c in out.columns if 'level' in c.lower() or 'type' in c.lower()]
    if candidate_cols:
        out['school_level_inferred'] = out[candidate_cols[0]]

    return out

## 8) 学校与公共数据质量过滤/审计

In [9]:
def quality_filter_public_data(schools_df: pd.DataFrame, hawker_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    school_audit = pd.DataFrame(columns=['reason'])
    hawker_audit = pd.DataFrame(columns=['reason'])

    out_schools = schools_df.copy()
    if len(out_schools):
        # keep rows with at least a probable school name column populated
        name_cols = [c for c in out_schools.columns if 'school' in c.lower() or 'name' in c.lower()]
        if name_cols:
            valid = out_schools[name_cols[0]].astype(str).str.len() > 0
            school_audit = out_schools[~valid].copy()
            school_audit['reason'] = 'missing_school_name'
            out_schools = out_schools[valid].copy()

    out_hawker = hawker_df.copy()
    if len(out_hawker) and {'lat', 'lng'}.issubset(out_hawker.columns):
        valid = out_hawker['lat'].notna() & out_hawker['lng'].notna()
        hawker_audit = out_hawker[~valid].copy()
        hawker_audit['reason'] = 'missing_coordinates'
        out_hawker = out_hawker[valid].copy()

    return out_schools, out_hawker, pd.concat([school_audit, hawker_audit], ignore_index=True)

## 9-11) 落盘、日志、完整性校验与一键执行单元

In [25]:
# Runtime tuning to avoid kernel hang during long OneMap collection
from urllib3.util.retry import Retry

REQUEST_TIMEOUT = (5, 10)
QPS_LIMIT = 3
MIN_INTERVAL = 1.0 / QPS_LIMIT
ONEMAP_MAX_ADDRESSES = 1500  # incremental batch; rerun cell 18 until pending reaches 0

adapter_fast = HTTPAdapter(
    max_retries=Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(['GET']),
    )
)
session.mount('http://', adapter_fast)
session.mount('https://', adapter_fast)

print('Runtime tuned:', {'REQUEST_TIMEOUT': REQUEST_TIMEOUT, 'QPS_LIMIT': QPS_LIMIT, 'ONEMAP_MAX_ADDRESSES': ONEMAP_MAX_ADDRESSES})

Runtime tuned: {'REQUEST_TIMEOUT': (5, 10), 'QPS_LIMIT': 3, 'ONEMAP_MAX_ADDRESSES': 1500}


In [33]:

RUN_PUBLIC_COLLECTION = True

run_ts = datetime.utcnow()
run_date = run_ts.strftime('%Y%m%d')
run_iso = run_ts.isoformat()

# 1) OneMap geocode for HDB addresses
if RUN_PUBLIC_COLLECTION:
    try:
        onemap_geo_raw, onemap_geo_audit = collect_onemap_hdb_geocode(address_df)
    except Exception as e:
        print(f'WARNING: OneMap geocoding failed: {e}')
        onemap_geo_raw, onemap_geo_audit = pd.DataFrame(), pd.DataFrame()
else:
    onemap_geo_raw, onemap_geo_audit = pd.DataFrame(), pd.DataFrame()

# Save geocoded data immediately so a later failure doesn't lose this result
onemap_geo_out = GEO_RAW_DIR / f'onemap_hdb_geocode_2015plus_{run_date}.csv'
if len(onemap_geo_raw):
    onemap_geo_raw.to_csv(onemap_geo_out, index=False)
    print(f'Saved {len(onemap_geo_raw)} geocoded rows → {onemap_geo_out.name}')

# 2) Public datasets: MOE schools + NEA hawkers + transit nodes
try:
    moe_schools_raw, moe_audit = collect_moe_school_data()
except Exception as e:
    print(f'WARNING: MOE school data failed: {e}')
    moe_schools_raw, moe_audit = pd.DataFrame(), pd.DataFrame()

try:
    nea_hawker_raw, nea_audit = collect_nea_hawker_data()
except Exception as e:
    print(f'WARNING: NEA hawker data failed: {e}')
    nea_hawker_raw, nea_audit = pd.DataFrame(), pd.DataFrame()

try:
    transit_nodes_raw = collect_onemap_transit_nodes()
except Exception as e:
    print(f'WARNING: Transit nodes failed: {e}')
    transit_nodes_raw = pd.DataFrame()

datamall_bus_stops = collect_datamall_bus_stops()

# 3) Normalize and quality filters
schools_processed = process_school_dataset(moe_schools_raw)
onemap_geo, transit_nodes, hawker_norm = normalize_geo(onemap_geo_raw, transit_nodes_raw, nea_hawker_raw)
schools_final, hawker_final, public_quality_audit = quality_filter_public_data(schools_processed, hawker_norm)

# 4) Save remaining outputs
transit_out = GEO_RAW_DIR / f'onemap_transit_nodes_{run_date}.csv'
hawker_out = GEO_RAW_DIR / f'nea_hawker_centres_{run_date}.csv'
bus_stop_out = GEO_RAW_DIR / f'datamall_bus_stops_{run_date}.csv'
schools_out = SCHOOL_RAW_DIR / f'moe_general_information_of_schools_{run_date}.csv'
audit_out = LOG_DIR / f'public_data_audit_{run_date}.csv'

if len(onemap_geo) and not onemap_geo_out.exists():
    onemap_geo.to_csv(onemap_geo_out, index=False)
if len(transit_nodes):
    transit_nodes.to_csv(transit_out, index=False)
if len(hawker_final):
    hawker_final.to_csv(hawker_out, index=False)
if len(datamall_bus_stops):
    datamall_bus_stops.to_csv(bus_stop_out, index=False)
if len(schools_final):
    schools_final.to_csv(schools_out, index=False)

audit_parts = []
for name, df in [
    ('onemap_geocode', onemap_geo_audit),
    ('moe_datastore', moe_audit),
    ('nea_datastore', nea_audit),
    ('quality', public_quality_audit),
]:
    if len(df):
        t = df.copy()
        t['audit_source'] = name
        audit_parts.append(t)

if audit_parts:
    pd.concat(audit_parts, ignore_index=True).to_csv(audit_out, index=False)

metadata = {
    'run_timestamp_utc': run_iso,
    'project_root': str(PROJECT_ROOT),
    'inputs': {
        'hdb_files': [str(p) for p in hdb_files],
        'hdb_rows_total': int(len(hdb_raw)),
        'hdb_rows_2015_plus': int(len(hdb_2015)),
        'address_count_2015_plus': int(len(address_df)),
    },
    'public_sources': {
        'one_map': {
            'api_key_present': bool(ONEMAP_API_KEY),
            'geocode_rows': int(len(onemap_geo)),
            'geocode_audit_rows': int(len(onemap_geo_audit)),
            'transit_nodes_rows': int(len(transit_nodes)),
        },
        'data_gov_sg': {
            'api_key_present': bool(DATA_GOV_SG_API_KEY),
            'moe_school_rows': int(len(schools_final)),
            'nea_hawker_rows': int(len(hawker_final)),
        },
        'lta_datamall': {
            'api_key_present': bool(DATAMALL_API_KEY),
            'bus_stop_rows': int(len(datamall_bus_stops)),
        },
    },
    'output_files': [
        str(onemap_geo_out), str(transit_out), str(hawker_out), str(bus_stop_out), str(schools_out), str(audit_out)
    ],
}

metadata_file = RAW_ROOT / f'raw_collection_metadata_{run_date}.json'
with metadata_file.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Integrity checks
if len(schools_final):
    assert len(schools_final) > 0, 'No school records after processing.'
if len(onemap_geo):
    assert onemap_geo['source_id'].is_unique, 'source_id not unique in OneMap geocode output'

print('=== Run Completed (Public Sources) ===')
print('Metadata:', metadata_file)
print('Raw outputs under:', RAW_ROOT)
for out_dir in [GEO_RAW_DIR, SCHOOL_RAW_DIR, LOG_DIR]:
    print('\n', out_dir)
    for p in sorted(out_dir.glob('*'))[-20:]:
        print(' -', p.name)

metadata


/var/folders/6p/nwg1ljcj77bfnrwnqw_yprf80000gn/T/ipykernel_66737/148734815.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_ts = datetime.utcnow()


Addresses pending for OneMap geocode: 0
All addresses already geocoded — reconstructing from JSONL...
Reconstructed 9710 geocoded rows from JSONL.
Saved 9710 geocoded rows → onemap_hdb_geocode_2015plus_20260316.csv
  [safe_get_json] Unexpected error for https://data.gov.sg/api/action/datastore_search: HTTPSConnectionPool(host='data.gov.sg', port=443): Max retries exceeded with url: /api/action/datastore_search?resource_id=d_688b934f82c1059ed0a6993d2a829089&limit=5000&offset=0 (Caused by ResponseError('too many 429 error responses'))
  [safe_get_json] Unexpected error for https://data.gov.sg/api/action/datastore_search: HTTPSConnectionPool(host='data.gov.sg', port=443): Max retries exceeded with url: /api/action/datastore_search?resource_id=d_4a086da0a5553be1d89383cd90d07ecd&limit=5000&offset=0 (Caused by ResponseError('too many 429 error responses'))
=== Run Completed (Public Sources) ===
Metadata: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_

{'run_timestamp_utc': '2026-03-16T05:49:17.644109',
 'project_root': '/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens',
 'inputs': {'hdb_files': ['/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv',
   '/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv',
   '/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/ResaleFlatPrices/Resale flat prices based on registration date from Jan-2017 onwards.csv'],
  'hdb_rows_total': 315627,
  'hdb_rows_2015_plus': 263424,
  'address_count_2015_plus': 9710},
 'public_sources': {'one_map': {'api_key_present': True,
   'geocode_rows': 9710,
   'geocode_audit_rows': 9710,
   'transit_nodes_rows

## 12) 增量增强：MRT、商圈通勤、道路噪音代理（不破坏主流程）

说明：
- MRT：从 OneMap transit nodes 中提取 MRT/LRT，并计算每个 HDB 地址到最近轨道站直线距离
- 商圈通勤：以 CBD 锚点估算最短直线距离与通勤时长代理
- 道路噪音代理：用与主干道/高速走廊的距离近似“面向马路噪音风险”
- 注：若后续拿到真实楼栋朝向字段，可替换该代理逻辑

In [34]:
# 增量特征增强（依赖 cell 19 的 onemap_geo / transit_nodes / run_date）
from math import radians, cos, sin, asin, sqrt

def _haversine_km(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return 6371.0 * c

def _nearest_distance_km(lat, lon, points):
    if pd.isna(lat) or pd.isna(lon) or len(points) == 0:
        return np.nan
    dmin = np.inf
    for p in points:
        d = _haversine_km(float(lat), float(lon), p[0], p[1])
        if d < dmin:
            dmin = d
    return float(dmin)

if 'onemap_geo' not in globals() or onemap_geo is None or len(onemap_geo) == 0:
    raise RuntimeError('onemap_geo 不存在或为空，请先执行一键采集单元（cell 19）。')

if 'transit_nodes' not in globals() or transit_nodes is None or len(transit_nodes) == 0:
    print('WARNING: transit_nodes 为空，尝试回退读取当日文件...')
    _fallback = sorted(GEO_RAW_DIR.glob('onemap_transit_nodes_*.csv'))
    transit_nodes = pd.read_csv(_fallback[-1]) if _fallback else pd.DataFrame()

# 基础地理表
geo_base = onemap_geo.copy()
for c in ['lat', 'lng']:
    geo_base[c] = pd.to_numeric(geo_base[c], errors='coerce')
geo_base = geo_base.dropna(subset=['lat', 'lng']).copy()

# 1) MRT/LRT 数据提取与最近站距离
mrt_nodes = transit_nodes.copy() if len(transit_nodes) else pd.DataFrame()
if len(mrt_nodes):
    if 'node_type' in mrt_nodes.columns:
        mrt_nodes = mrt_nodes[mrt_nodes['node_type'].astype(str).str.contains('mrt|lrt', case=False, na=False)].copy()
    for c in ['lat', 'lng']:
        mrt_nodes[c] = pd.to_numeric(mrt_nodes[c], errors='coerce')
    mrt_nodes = mrt_nodes.dropna(subset=['lat', 'lng']).drop_duplicates(subset=['searchval', 'lat', 'lng']).copy()

mrt_points = list(zip(mrt_nodes['lat'], mrt_nodes['lng'])) if len(mrt_nodes) else []
geo_base['nearest_mrt_km'] = geo_base.apply(lambda r: _nearest_distance_km(r['lat'], r['lng'], mrt_points), axis=1)
geo_base['nearest_mrt_walk_min_proxy'] = (geo_base['nearest_mrt_km'] * 1000 / 75).round(1)  # ~75m/min

# 2) 商圈通勤代理（CBD anchors）
CBD_ANCHORS = [
    ('Raffles Place', 1.2846, 103.8518),
    ('Marina Bay', 1.2764, 103.8547),
    ('Tanjong Pagar', 1.2769, 103.8459),
    ('Orchard', 1.3048, 103.8318),
]
cbd_points = [(x[1], x[2]) for x in CBD_ANCHORS]
geo_base['cbd_min_km'] = geo_base.apply(lambda r: _nearest_distance_km(r['lat'], r['lng'], cbd_points), axis=1)

# 通勤时长代理：可按需求调整速度假设
# 车速 28 km/h，公共交通综合 22 km/h
geo_base['cbd_drive_min_proxy'] = (geo_base['cbd_min_km'] / 28 * 60).round(1)
geo_base['cbd_transit_min_proxy'] = (geo_base['cbd_min_km'] / 22 * 60).round(1)

# 3) 道路噪音代理（主干道/高速走廊点集）
MAJOR_ROAD_POINTS = [
    (1.3000, 103.9200),  # ECP/KPE corridor
    (1.2900, 103.8600),  # MCE/Marina corridor
    (1.3200, 103.8000),  # PIE/CTE corridor
    (1.3400, 103.7500),  # AYE west corridor
    (1.3600, 103.9000),  # TPE/KPE north-east corridor
    (1.3550, 103.7000),  # BKE/KJE corridor
    (1.3250, 103.8800),  # KPE/PIE east-central corridor
]
geo_base['major_road_min_km'] = geo_base.apply(lambda r: _nearest_distance_km(r['lat'], r['lng'], MAJOR_ROAD_POINTS), axis=1)

# 噪音风险分级：距离越近风险越高
geo_base['road_noise_risk'] = pd.cut(
    geo_base['major_road_min_km'],
    bins=[-np.inf, 0.25, 0.6, 1.2, np.inf],
    labels=['very_high', 'high', 'medium', 'low']
).astype(str)

# 以 0-1 连续分数表达（越大越吵）
geo_base['road_noise_score'] = (1 / (1 + geo_base['major_road_min_km'])).clip(0, 1).round(4)

# 注意：当前无可靠 unit 朝向字段，以下为“面向马路概率代理”
geo_base['unit_orientation_known'] = False
geo_base['facing_road_noise_proxy'] = np.where(
    geo_base['road_noise_risk'].isin(['very_high', 'high']), 'likely_facing_or_exposed', 'likely_less_exposed'
 )

# 输出文件
_date = run_date if 'run_date' in globals() else datetime.utcnow().strftime('%Y%m%d')
mrt_out = GEO_RAW_DIR / f'onemap_mrt_lrt_nodes_{_date}.csv'
features_out = GEO_RAW_DIR / f'hdb_geo_accessibility_noise_features_{_date}.csv'

if len(mrt_nodes):
    mrt_nodes.to_csv(mrt_out, index=False)
geo_base.to_csv(features_out, index=False)

print('=== 增量增强完成 ===')
print('MRT/LRT nodes:', len(mrt_nodes), '->', mrt_out.name if len(mrt_nodes) else 'N/A')
print('Geo feature rows:', len(geo_base), '->', features_out.name)
print('Columns added:', [
    'nearest_mrt_km', 'nearest_mrt_walk_min_proxy',
    'cbd_min_km', 'cbd_drive_min_proxy', 'cbd_transit_min_proxy',
    'major_road_min_km', 'road_noise_risk', 'road_noise_score',
    'unit_orientation_known', 'facing_road_noise_proxy'
])

=== 增量增强完成 ===
MRT/LRT nodes: 94 -> onemap_mrt_lrt_nodes_20260316.csv
Geo feature rows: 9710 -> hdb_geo_accessibility_noise_features_20260316.csv
Columns added: ['nearest_mrt_km', 'nearest_mrt_walk_min_proxy', 'cbd_min_km', 'cbd_drive_min_proxy', 'cbd_transit_min_proxy', 'major_road_min_km', 'road_noise_risk', 'road_noise_score', 'unit_orientation_known', 'facing_road_noise_proxy']


In [23]:

# ─────────────────────────────────────────────────────────────────────────
# REPAIR STEP: force-reload token, verify, clean JSONL + checkpoint
# Run this once, then run cell 19 repeatedly until all addresses are done.
# ─────────────────────────────────────────────────────────────────────────
import json as _json, shutil as _shutil

# Step 1 – force-reload .env (override existing env vars)
_env_path = PROJECT_ROOT / '01_data_layer' / '.env'
for _line in _env_path.read_text(encoding='utf-8').splitlines():
    _line = _line.strip()
    if _line and not _line.startswith('#') and '=' in _line:
        _k, _v = _line.split('=', 1)
        os.environ[_k.strip()] = _v.strip().strip('"').strip("'")

ONEMAP_API_KEY = os.environ.get('ONEMAP_API_KEY', '')
print(f"Token present: {bool(ONEMAP_API_KEY)}  length: {len(ONEMAP_API_KEY)}")

# Step 2 – verify token with a stripped address (no ", Singapore")
import requests as _req
_test_addr = re.sub(r',?\s*Singapore\s*$', '', address_df.iloc[0]['address_query'], flags=re.IGNORECASE).strip()
_r = _req.get(ONEMAP_SEARCH_ENDPOINT,
               params={'searchVal': _test_addr, 'returnGeom': 'Y', 'getAddrDetails': 'Y', 'pageNum': 1},
               headers={'Authorization': ONEMAP_API_KEY}, timeout=10)
_p = _r.json()
_found = _p.get('found', 0)
_err = _p.get('error')
print(f"Token test ('{_test_addr}') → found={_found}  error={_err}")
assert _found > 0, f"Token not working! error={_err}. Renew at onemap.gov.sg → update 01_data_layer/.env"
print("✓ Token is valid")

# Step 3 – keep only successfully-geocoded JSONL entries, discard invalid ones
_ok_lines = []
_ok_ids = set()
with ONEMAP_GEO_RAW_RESP_FILE.open('r', encoding='utf-8') as _f:
    for _line in _f:
        _line = _line.strip()
        if not _line:
            continue
        try:
            _obj = _json.loads(_line)
            if _obj.get('response', {}).get('payload', {}).get('results', []):
                _sid = str(_obj.get('source_id', ''))
                if _sid not in _ok_ids:  # deduplicate
                    _ok_ids.add(_sid)
                    _ok_lines.append(_line)
        except Exception:
            pass

# Back up and rewrite JSONL with only valid entries
_bak = ONEMAP_GEO_RAW_RESP_FILE.with_suffix('.jsonl.bak')
_shutil.copy2(ONEMAP_GEO_RAW_RESP_FILE, _bak)
with ONEMAP_GEO_RAW_RESP_FILE.open('w', encoding='utf-8') as _f:
    for _line in _ok_lines:
        _f.write(_line + '\n')
print(f"JSONL cleaned: kept {len(_ok_lines)} valid entries (backup: {_bak.name})")

# Step 4 – reset checkpoint to only the valid IDs
_clean_ckpt = pd.DataFrame({'source_id': sorted(_ok_ids)})
_clean_ckpt.to_csv(ONEMAP_GEO_CHECKPOINT_FILE, index=False)
print(f"Checkpoint reset to {len(_clean_ckpt)} rows"
      f" → {len(address_df) - len(_clean_ckpt)} addresses still pending")
print("\nReady — run cell 19 repeatedly until 'Addresses pending' reaches 0.")


Token present: True  length: 572
Token test ('174 ANG MO KIO AVE 4') → found=1  error=None
✓ Token is valid
JSONL cleaned: kept 10 valid entries (backup: onemap_geocode_raw_responses.jsonl.bak)
Checkpoint reset to 10 rows → 9700 addresses still pending

Ready — run cell 19 repeatedly until 'Addresses pending' reaches 0.


In [22]:

# Token & address format quick diagnostic
import requests as _req

_ep = ONEMAP_SEARCH_ENDPOINT

# Force reload fresh token from .env
_env_path = PROJECT_ROOT / '01_data_layer' / '.env'
for _line in _env_path.read_text(encoding='utf-8').splitlines():
    _line = _line.strip()
    if _line and not _line.startswith('#') and '=' in _line:
        _k, _v = _line.split('=', 1)
        os.environ[_k.strip()] = _v.strip().strip('"').strip("'")
ONEMAP_API_KEY = os.environ.get('ONEMAP_API_KEY', '')
_hdrs = {'Authorization': ONEMAP_API_KEY}

for _q in ['174 ANG MO KIO AVE 4', '174 ANG MO KIO AVE 4, Singapore', 'TAMPINES', 'BLK 1 BEDOK NORTH AVE 4']:
    _r = _req.get(_ep, params={'searchVal': _q, 'returnGeom': 'Y', 'getAddrDetails': 'Y', 'pageNum': 1}, headers=_hdrs, timeout=10)
    _p = _r.json()
    print(f"[WITH token] '{_q}' → found={_p.get('found', 'N/A')}  error={_p.get('error')}")

# Without token
_r0 = _req.get(_ep, params={'searchVal': 'TAMPINES', 'returnGeom': 'Y', 'getAddrDetails': 'Y', 'pageNum': 1}, timeout=10)
_p0 = _r0.json()
print(f"\n[NO token]   'TAMPINES' → found={_p0.get('found', 'N/A')}  error={_p0.get('error')}")


[WITH token] '174 ANG MO KIO AVE 4' → found=1  error=None
[WITH token] '174 ANG MO KIO AVE 4, Singapore' → found=0  error=None
[WITH token] 'TAMPINES' → found=661  error=None
[WITH token] 'BLK 1 BEDOK NORTH AVE 4' → found=2  error=None

[NO token]   'TAMPINES' → found=661  error=Authentication token missing. Please create an account and generate or renew your API Token.
